# Learning from enzyme–substrate kinetics
This CPU notebook uses synthetic Michaelis–Menten curves. The parent workbench covers measured Puromycin rates. Run this notebook from the ml directory after installing requirements.txt and scikit-learn.

In [1]:
import numpy as np
import pandas as pd
import molecular as lab

params, rates, families = lab.simulate(2400, 17)
truth, test, test_families = lab.simulate(800, 91)
print(pd.DataFrame(rates, columns=lab.DOSES).head().round(1).to_string(index=False))

 0.02  0.06  0.11  0.22  0.56  1.10
  5.3  33.4  36.0  72.5  93.8 124.4
 10.9  16.6  67.0  96.5 137.9 148.2
 66.2  89.2  94.2 107.8 143.1 138.8
 53.9 104.1 101.1 140.5 149.1 132.4
 46.1 108.8 128.0 144.2 191.2 204.4


## Supervised parameter inference
Both estimators see the six reaction rates. The PyTorch MLP minimizes mean squared error on scaled Vmax and log Km, then reverses those transforms before evaluation.

In [2]:
forest, net = lab.train_regressors(rates, params)
errors = {}
for name, estimator in [('Random forest', forest), ('PyTorch MLP', net)]:
    predicted = estimator(test)
    vmax, km = predicted[:, 0]*180, .1*np.exp(predicted[:,1])
    errors[name] = np.abs(np.column_stack([vmax,km])-truth).mean(axis=0)
print(pd.DataFrame.from_dict(errors, orient='index', columns=['Vmax MAE','Km MAE']).round(4))

               Vmax MAE  Km MAE
Random forest    9.8111  0.0215
PyTorch MLP     11.3304  0.0232


## One-step reinforcement learning
The state is the binned rate measured at 0.11 ppm. The action chooses a second concentration and the terminal reward comes from predicting the correct synthetic family. A policy gains credit only for predictions from the same decoder and measurement budget as the baselines.

In [3]:
_, episodes, labels = lab.simulate(4800, 42)
rewards = np.column_stack([lab.decoder(rates,families,episodes,a)==labels for a in lab.ACTIONS])
q = lab.train_policy(np.digitize(episodes[:,lab.ANCHOR],lab.BINS),rewards)
scores = np.column_stack([lab.decoder(rates,families,test,a)==test_families for a in lab.ACTIONS])
states = np.digitize(test[:,lab.ANCHOR],lab.BINS)
policy = scores[np.arange(len(test)),q[states].argmax(axis=1)]
fixed = scores[:,rewards.mean(axis=0).argmax()]
print('Learned, fixed, random:', policy.mean(), fixed.mean(), scores.mean())
paired = policy.astype(float)-fixed
boot = np.random.default_rng(73).choice(paired,size=(2000,len(paired))).mean(axis=1)
print('Paired 95% bootstrap interval in percentage points:', 100*np.quantile(boot,[.025,.975]))

Learned, fixed, random: 0.83375 0.82375 0.76875
Paired 95% bootstrap interval in percentage points: [-0.125  2.125]
